In [2]:
import pandas as pd
import joblib

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV
)

from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline

from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.ensemble import (
    RandomForestClassifier,
    AdaBoostClassifier,
    GradientBoostingClassifier
)

from sklearn.metrics import accuracy_score

from xgboost import XGBClassifier

In [3]:
# Load Dataset
df = pd.read_csv("heart.csv")

# Display first 5 rows
print(df.head())

# Dataset Information
print(df.info())

# Check Missing Values
print(df.isnull().sum())

   Age Sex ChestPainType  RestingBP  Cholesterol  FastingBS RestingECG  MaxHR  \
0   40   M           ATA        140          289          0     Normal    172   
1   49   F           NAP        160          180          0     Normal    156   
2   37   M           ATA        130          283          0         ST     98   
3   48   F           ASY        138          214          0     Normal    108   
4   54   M           NAP        150          195          0     Normal    122   

  ExerciseAngina  Oldpeak ST_Slope  HeartDisease  
0              N      0.0       Up             0  
1              N      1.0     Flat             1  
2              N      0.0       Up             0  
3              Y      1.5     Flat             1  
4              N      0.0       Up             0  
<class 'pandas.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             91

In [8]:
X = df.drop("HeartDisease", axis=1)
y = df["HeartDisease"]

# Convert categorical columns into numeric
X = pd.get_dummies(X, drop_first=True)

print(X.head())

   Age  RestingBP  Cholesterol  FastingBS  MaxHR  Oldpeak  Sex_M  \
0   40        140          289          0    172      0.0   True   
1   49        160          180          0    156      1.0  False   
2   37        130          283          0     98      0.0   True   
3   48        138          214          0    108      1.5  False   
4   54        150          195          0    122      0.0   True   

   ChestPainType_ATA  ChestPainType_NAP  ChestPainType_TA  RestingECG_Normal  \
0               True              False             False               True   
1              False               True             False               True   
2               True              False             False              False   
3              False              False             False               True   
4              False               True             False               True   

   RestingECG_ST  ExerciseAngina_Y  ST_Slope_Flat  ST_Slope_Up  
0          False             False          F

In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [11]:
k_values = [3,5,7,9,11,13]

best_k = 0
best_score = 0

print("Manual Search (KNN)\n")

for k in k_values:

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("knn", KNeighborsClassifier(n_neighbors=k))
    ])

    model.fit(X_train,y_train)

    score = model.score(X_test,y_test)

    print(f"K = {k}  Accuracy = {score:.4f}")

    if score > best_score:
        best_score = score
        best_k = k

print("\nBest K :",best_k)
print("Best Accuracy :",round(best_score,4))

Manual Search (KNN)

K = 3  Accuracy = 0.9022
K = 5  Accuracy = 0.8859
K = 7  Accuracy = 0.8967
K = 9  Accuracy = 0.9022
K = 11  Accuracy = 0.8967
K = 13  Accuracy = 0.8859

Best K : 3
Best Accuracy : 0.9022


In [12]:
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("svm",SVC())
])

param_grid = {
    "svm__C":[1,10,20],
    "svm__kernel":["linear","rbf"]
}

grid = GridSearchCV(
    pipe,
    param_grid,
    cv=5
)

grid.fit(X_train,y_train)

print(grid.best_params_)
print(grid.best_score_)

{'svm__C': 1, 'svm__kernel': 'rbf'}
0.8596123380859193


In [13]:
random = RandomizedSearchCV(
    pipe,
    param_grid,
    n_iter=5,
    cv=5,
    random_state=42
)

random.fit(X_train,y_train)

print(random.best_params_)
print(random.best_score_)

{'svm__kernel': 'rbf', 'svm__C': 1}
0.8596123380859193


In [15]:
results = []

# SVM
svm = grid.best_estimator_
svm.fit(X_train,y_train)

pred = svm.predict(X_test)

results.append([
    "SVM",
    accuracy_score(y_test,pred)
])

# Random Forest

rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train,y_train)

pred = rf.predict(X_test)

results.append([
    "Random Forest",
    accuracy_score(y_test,pred)
])

# AdaBoost

ada = AdaBoostClassifier(
    n_estimators=100,
    random_state=42
)

ada.fit(X_train,y_train)

pred = ada.predict(X_test)

results.append([
    "AdaBoost",
    accuracy_score(y_test,pred)
])

# Gradient Boosting

gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

gb.fit(X_train,y_train)

pred = gb.predict(X_test)

results.append([
    "Gradient Boosting",
    accuracy_score(y_test,pred)
])

# XGBoost

xgb = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train,y_train)

pred = xgb.predict(X_test)

results.append([
    "XGBoost",
    accuracy_score(y_test,pred)
])

comparison = pd.DataFrame(
    results,
    columns=["Model","Accuracy"]
)

print(comparison)

best_model = comparison.loc[
    comparison["Accuracy"].idxmax()
]

print("\nBest Model\n")
print(best_model)

               Model  Accuracy
0                SVM  0.902174
1      Random Forest  0.875000
2           AdaBoost  0.891304
3  Gradient Boosting  0.875000
4            XGBoost  0.896739

Best Model

Model            SVM
Accuracy    0.902174
Name: 0, dtype: object


In [16]:
joblib.dump(svm,"best_heart_model.pkl")
columns = X.columns.tolist()

joblib.dump(columns, "columns.pkl")

print("Model Saved Successfully")

Model Saved Successfully
